# Word Topic Modelling

In [1]:
!pip install bertopic datasets accelerate bitsandbytes>=0.46.1 xformers adjustText

In [2]:
from datasets import load_dataset

dataset =  load_dataset("CShorten/ML-ArXiv-Papers")["train"]

abstracts = dataset["abstract"]
titles = dataset["title"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [3]:
# The abstract of "Attention is All You Need"
print(abstracts[13894])

  The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks in an encoder-decoder configuration. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer, based
solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to be
superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014
English-to-German translation task, improving over the existing best results,
including ensembles by over 2 BLEU. On the WMT 2014 English-to-French
translation task, our model establishes a new single-model state-of-the-art
BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction
of the training costs of the best models from the literature. We show that the
Transfor

In [4]:
from torch import cuda

model_id = "NousResearch/LLaMA-2-7b-chat-hf"
device = "cuda" if cuda.is_available() else "cpu"

print(f"Using device: {device}")

Using device: cuda


## Quantising the Model to 4 bits

In [5]:
from torch import bfloat16
import transformers

bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16
)

## Downloading the Model tokensizer and Model

In [6]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_id)

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto",
)


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [7]:
generator = transformers.pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.1,
    max_new_tokens=512,
    repetition_penalty=1.1
)

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [8]:
system_prompt = """
<s>[INST] <<SYS>>
You are a helpful, respectful and honest assistant for labeling topics
<</SYS>>
"""

In [10]:
example_prompt = """
System Instructions: You are a precise text-classification assistant. Analyze the three provided documents and the important keywords extracted from them. 
Based on this information, create a single short label for the overarching topic. You must return the label and nothing else—no explanations, no preamble, and no extra text.Document 

1: Machine learning algorithms now analyze medical imaging to detect anomalies like tumors with a level of accuracy that rivals experienced radiologists.Document 
2: Neural networks process vast genomic datasets to identify patient-specific gene mutations and predict cellular responses to targeted therapies.Document 
3: Automated robotic tools assist surgeons by executing micro-movements during complex procedures, minimizing invasive tissue damage.

Important Keywords Used: Machine learning, algorithms, medical imaging, radiologistsNeural networks, genomic, mutations, therapiesRobotic tools, surgeons, procedures, tissue damage
Instruction: Based on the above information, please create a short label of this topic. Return the label and label only.
"""

In [11]:
main_prompt = """
[INST]
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: '[KEYWORDS]'.

Based on the information about the topic aboce, plase create a short label of this topic.
Make sure you only return the label and nothing more
[/INST]
"""

In [12]:
prompt = system_prompt + example_prompt + main_prompt

In [13]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("BAAI/bge-small-en", device=device)
embeddings = embedding_model.encode(abstracts, convert_to_tensor=True, show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/90.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3675 [00:00<?, ?it/s]

In [14]:
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(
    n_neighbors=15,
    n_components=5, 
    min_dist=0.0, 
    metric='cosine', 
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=150,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

In [16]:
# ensure embeddings is a NumPy array on CPU
emb_np = embeddings.detach().cpu().numpy() if hasattr(embeddings, "detach") else embeddings
reduce_embeddings = umap_model.fit_transform(emb_np)

In [ ]:
from bertopic.representation import MaximalMarginalRelevance, KeyBERTInspired, TopicGeneration

keybert = KeyBERTInspired()

mmr = MaximalMarginalRelevance(diversity=0.3)

llama2 = TextGeneration(generator, prompt=prompt)

representation_model = {
    "KeyBERT": keybert,
    "Llama2": llama2,
    "MMR": mmr
}

In [ ]:
from bertopic import BERTopic 

topic_model = BERTopic(

    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    representation_model=representation_model,

    top_n_words=10,
    verbose=True
)

topics, probs = topic_model.fit_transform(abstracts, embeddings=embeddings)

In [ ]:
topic_model.get_topic_info()